In [2]:
import os
from nwtrace import *
import pandas as pd
import geopandas as gpd

from pathlib import Path


In [3]:
lines = Path('data/more/full_sewers.geojson')
nodes = Path('data/more/all_node_connections.geojson')

lines_gdf = gpd.read_file(lines)
nodes_gdf = gpd.read_file(nodes)

nodes_gdf = nodes_gdf.to_crs(lines_gdf.crs)


In [4]:
multiple = True
upstream_only = False
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROMMH'
downstream_field = 'TOMH'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls

outputname_extra = "allBC_"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

sewershed = NWTrace(
    network=lines_gdf,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

# additional connections
nodes_up = (nodes_gdf[["FACILITYID", "TO_ASSET_ID"]]
            .dropna(subset=["FACILITYID", "TO_ASSET_ID"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_ASSET_ID": 'segment_id'})
            .to_dict(orient="records"))

sewershed.add_upstream_nodes(nodes_up)


node_lookup, seg_lookup = sewershed.get_lookup_tables()
dir_node_lookup, dir_seg_lookup = sewershed.get_directional_lookup_tables()


Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 4217 new segment(s).



In [5]:
errors = utils.verify_network_geometry(
    lines=lines_gdf,
    points=nodes_gdf,
    segment_lookup=seg_lookup,
    line_id_field="FACILITYID",
    point_id_field="FACILITYID",
    threshold=100
)

err_df = gpd.GeoDataFrame.from_dict(errors, geometry="geometry")
err_df = err_df.set_crs(lines_gdf.crs)

err_df.to_file("out/errors.gpkg", driver="GPKG", layer="errorsv1")

err_df.value_counts("error_t")

100%|██████████| 173166/173166 [00:02<00:00, 64668.96it/s]


5014 Errors Found
Average Distance 0.11152260733136245 units


error_t
missing segment    4217
missing node        789
spatial               8
Name: count, dtype: int64

In [6]:
ids_dup = utils.count_duplicates(lines_gdf, "FACILITYID", minimum_count=1)
ids_dup

{'duplicate_count': {}}

In [8]:
segments_fixed = utils.repair_spatial_errors(
    errors=errors,
    segments=lines_gdf,
    nodes=nodes_gdf,
    line_id_field="FACILITYID",
    upstream_field="FROMMH",
    downstream_field="TOMH",
    distance_threshold=1
)

segments_fixed.reset_index(drop=True, inplace=True)
segments_fixed.to_file("out/repaired_sewers.gpkg")

In [ ]:
nodes_fixed = utils.repair_node_connections(
    errors=errors,
    nodes=nodes_gdf,
    segments=lines_gdf,
    node_id_field="FACILITYID",
    segment_id_field="FACILITYID",
    node_connection_field="TO_ASSET_ID",
    upstream_field="FROMMH",
    downstream_field="TOMH",
)

nodes_fixed.reset_index(drop=True, inplace=True)
nodes_fixed.to_file("out/repaired_nodes.gpkg")

NameError: name 'utils' is not defined

In [ ]:

# Regenerate lookup tables with the repaired dataset
fixed_sewershed = NWTrace(
    network=segments_fixed,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

# additional connections
# additional connections
nodes_up = (nodes_fixed[["FACILITYID", "TO_ASSET_ID"]]
            .dropna(subset=["FACILITYID", "TO_ASSET_ID"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_ASSET_ID": 'segment_id'})
            .to_dict(orient="records"))

fixed_sewershed.add_upstream_nodes(nodes_up)


f_node_lookup, f_seg_lookup = fixed_sewershed.get_lookup_tables()
f_dir_node_lookup, f_dir_seg_lookup = fixed_sewershed.get_directional_lookup_tables()

Added 5246 node-segment connection(s)
Created 627 new node(s)
Created 4217 new segment(s).



In [ ]:
# Check the amount of spatial errors in the newly repaired dataset
f_errors = utils.verify_network_geometry(
    lines=segments_fixed,
    points=nodes_gdf,
    segment_lookup=f_seg_lookup,
    line_id_field="FACILITYID",
    point_id_field="FACILITYID",
    threshold=100
)

f_err_df = gpd.GeoDataFrame.from_dict(f_errors, geometry="geometry")
f_err_df = f_err_df.set_crs(lines_gdf.crs)

f_err_df.value_counts("error_t")

100%|██████████| 173166/173166 [00:02<00:00, 61890.55it/s]

5006 Errors Found
Average Distance 0.0025141119628859024 units


error_t
missing segment    4217
missing node        789
Name: count, dtype: int64

In [ ]:
f_err_df[f_err_df["error_t"] == "spatial"]

,node_id,segment_id,error_t,error_msg,dist,geometry
